# Pose Image Tool

참조 이미지의 자세(pose)를 유지한 채 다른 인물/장면을 생성합니다.

- Base model: `stabilityai/sdxl-turbo`
- ControlNet: `thibaud/controlnet-openpose-sdxl-1.0`
- Steps: 4 | Guidance: 0.0

## 1. Install dependencies

In [ ]:
!pip install -q diffusers transformers accelerate controlnet_aux opencv-python

## 2. Download reference image

자세를 참조할 사람 사진을 준비합니다. (자체 업로드도 가능)

In [ ]:
import requests

urls = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Man_standing_with_arms_crossed_%2846218008430%29.jpg/600px-Man_standing_with_arms_crossed_%2846218008430%29.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Young_man_standing_in_city.jpg/600px-Young_man_standing_in_city.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/2/20/Man_standing_%28cropped%29.jpg/600px-Man_standing_%28cropped%29.jpg",
]

for url in urls:
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            with open("ref.jpg", "wb") as f:
                f.write(r.content)
            print(f"Downloaded: {url.split('/')[-2]}")
            break
    except:
        continue
else:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        !mv "{name}" ref.jpg
        break

from IPython.display import Image as DImage, display
display(DImage("ref.jpg"))

## 3. OpenPose keypoint extraction

In [ ]:
from controlnet_aux import OpenposeDetector
from PIL import Image

ref = Image.open("ref.jpg").convert("RGB")
print(f"Reference size: {ref.size}")

detector = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")
pose = detector(ref, hand_and_face=True)

pose.save("pose.png")
display(pose)

## 4. Load SDXL + ControlNet pipeline

In [ ]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, AutoencoderKL

controlnet = ControlNetModel.from_pretrained(
    "thibaud/controlnet-openpose-sdxl-1.0",
    torch_dtype=torch.float16,
)

vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch.float16,
)

pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/sdxl-turbo",
    controlnet=controlnet,
    vae=vae,
    torch_dtype=torch.float16,
    variant="fp16",
)
pipe.to("cuda")
pipe.enable_model_cpu_offload()

print("Pipeline loaded")

## 5. Generate

프롬프트를 원하는 대로 수정하고 실행하세요.

In [ ]:
prompt = "a young woman in a red dress, cinematic lighting, detailed face, professional photo"

result = pipe(
    prompt=prompt,
    negative_prompt="bad anatomy, ugly, disfigured",
    image=pose,
    num_inference_steps=4,
    guidance_scale=0.0,
    controlnet_conditioning_scale=0.7,
    generator=torch.Generator(device="cuda").manual_seed(42),
).images[0]

result.save("output.png")
display(result)

## 6. Compare reference and result

In [ ]:
ref = Image.open("ref.jpg").convert("RGB")
res = Image.open("output.png")

w, h = ref.size
resized = res.resize((w, h))

display(ref)
display(resized)